# 03 - Naive OLS

Simple regression of traffic fatalities on legalization status with no fixed effects. Establishes the biased baseline before adding design.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.dpi"] = 120

DATA_DIR = Path("../data/processed")
OUT_DIR  = Path("../outputs")
OUT_DIR.mkdir(exist_ok=True)

FARS_FILE = "fars_state_year.parquet"
CDC_FILE  = "cdc_state_year.parquet"

# Load FARS panel
if not (DATA_DIR / FARS_FILE).exists():
    raise FileNotFoundError(
        f"{FARS_FILE} not found. Run:\n"
        "  python scripts/download_fars.py\n"
        "  python src/build_fars_panel.py"
    )
fars = pd.read_parquet(DATA_DIR / FARS_FILE)
leg  = pd.read_csv("../data/codebooks/state_legalization_dates.csv")
print(f"FARS panel: {fars.shape}  |  States: {fars['state'].nunique()}  |  Years: {sorted(fars['year'].dropna().unique()[:3])}...{sorted(fars['year'].dropna().unique()[-3:])}")

In [ ]:
import statsmodels.formula.api as smf

In [ ]:
primary = "total_fatalities_per_100k" if "total_fatalities_per_100k" in fars.columns else "total_fatalities"

# Add treated indicator
fars_reg = fars.merge(leg[['state','retail_sales_year']], on='state', how='left')
fars_reg['ever_treated'] = fars_reg['retail_sales_year'].notna().astype(int)
fars_reg['post'] = (fars_reg['retail_sales_year'].notna() & (fars_reg['year'] >= fars_reg['retail_sales_year'])).astype(int)
fars_reg['did'] = fars_reg['ever_treated'] * fars_reg['post']

## Model 1 - Simple cross-sectional (most biased)

In [ ]:
m1 = smf.ols(f"{primary} ~ ever_treated", data=fars_reg).fit(cov_type='HC3')
print(f"Treated vs never-treated: β = {m1.params['ever_treated']:.4f}")
print("This is pure selection — treated states differ in many ways from control states.")

## Model 2 - Before/after for treated states only (no controls)

In [ ]:
m2 = smf.ols(f"{primary} ~ post", data=fars_reg[fars_reg['ever_treated']==1]).fit(cov_type='HC3')
print(f"Before/after (treated only): β = {m2.params['post']:.4f}")
print("Confounds legalization with any other trend in treated states over time.")

## Model 3 - Basic DiD (no FE)

In [ ]:
m3 = smf.ols(f"{primary} ~ ever_treated + post + did", data=fars_reg).fit(cov_type='HC3')
att = m3.params['did']
ci  = m3.conf_int().loc['did']
print(f"Basic DiD ATT: β = {att:.4f}  95% CI [{ci[0]:.4f}, {ci[1]:.4f}]")
print("\nProblem: pooled DiD with staggered treatment timing is biased.")
print("Treated units in earlier cohorts act as controls for later cohorts.")
print("See notebooks 04 (TWFE) and 05 (Callaway-Sant'Anna) for better estimators.")